# Boosted Decision Tree

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, make_scorer
import tabulate
from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')
# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training


In [ ]:
def training(file_path, csv_name):
    # Leggo i csv
    df = pd.read_csv(file_path)
    
    # Mi definisco la lista dei target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']
    
    # Filtro solo le pazienti con PR valido
    df_validi = df.dropna(subset=original_target_list).copy()
    
    # Trasformo tutto in valori binari
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']
    
    # Definisco features
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')
    
    target = df_validi[final_target_list]
    groups = df_validi['Patient ID']
    
    # Riempio Nan (Anche se HistGradientBoosting gestisce NaN nativamente, è buona norma se ci sono colonne tutte vuote)
    features = features.fillna(features.mean())
    
    # Cross Validation
    cv = GroupKFold(n_splits=5)
    
    # Modello Base: HistGradientBoostingClassifier
    # n_jobs=1 qui per parallelizzare la GridSearch
    base_model = HistGradientBoostingClassifier(random_state=42) 
    multi_output_model = MultiOutputClassifier(base_model)
    
    # Iperparametri (prefisso estimator__ obbligatorio)
    iperparametri = {
        'estimator__learning_rate': [0.05, 0.1],
        'estimator__max_iter': [100],
        'estimator__max_depth': [10, None],
        'estimator__min_samples_leaf': [5],
        'estimator__l2_regularization': [0, 0.1],
        'estimator__max_leaf_nodes': [31],
        'estimator__early_stopping': ['auto'] # Nota: early_stopping richiede validation fraction, default auto va bene
    }
    
    # Scorer Custom (Safe)
    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0))
        return np.mean(scores)
    
    scorer = make_scorer(multi_f1_scorer)
    
    # Calcolo combinazioni
    total_combinations = math.prod(len(v) for v in iperparametri.values())
    print(f"\nInizio Grid Search ({total_combinations} combinazioni) per: {csv_name}")
    
    # Grid Search
    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=cv,
        scoring=scorer,
        n_jobs=-1,
        verbose=1,
        refit=False,
        error_score='raise'
    )
    
    # Esecuzione
    grid_search.fit(features, target, groups=groups)
    
    # --- Output Format ---
    scores = []
    results = grid_search.cv_results_
    
    for i in range(len(results['params'])):
        # Pulisco chiavi
        params = {k.replace('estimator__', ''): v for k, v in results['params'][i].items()}
        
        current_fold_scores = [results[f'split{k}_test_score'][i] for k in range(5)]
        
        scores.append({
            **params,
            'mean_score': results['mean_test_score'][i],
            'std_score': results['std_test_score'][i],
            'fold_scores': current_fold_scores
        })
        
    return scores

# Vado a stampare gli output in una maniera piú leggibile

In [ ]:
def print_results(results_per_dataset):
    '''
    Stampa i risultati della Grid Search in modo organizzato usando tabulate
    SPECIFICO PER HIST GRADIENT BOOSTING
    '''
    print("\n" + "=" * 80)
    print(" " * 20 + "RIEPILOGO DEI MIGLIORI RISULTATI")
    print("=" * 80)

    summary_data = []

    for name, metrics_list in results_per_dataset.items():
        best_result = max(metrics_list, key=lambda x: x['mean_score'])

        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")

        # Tabella Iperparametri
        print("Iperparametri Ottimali:")
        params_table = [
            ['learning_rate', best_result.get('learning_rate')],
            ['max_iter', best_result.get('max_iter')],
            ['max_depth', best_result.get('max_depth')],
            ['min_samples_leaf', best_result.get('min_samples_leaf')],
            ['l2_regularization', best_result.get('l2_regularization')],
            ['max_leaf_nodes', best_result.get('max_leaf_nodes')],
            ['early_stopping', best_result.get('early_stopping')]
        ]
        print(tabulate.tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))
        print()

        # Aggiungi al riepilogo comparativo
        summary_data.append([
            name,
            best_result['mean_score'], # Tengo float per ordinamento
            best_result['std_score'],
            best_result.get('learning_rate'),
            best_result.get('max_depth'),
            best_result.get('l2_regularization'),
            best_result.get('max_iter')
        ])

    # Riepilogo Comparativo Finale
    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO TRA TUTTI I DATASET")
    print("=" * 80 + "\n")

    # Ordina per F1-score decrescente
    summary_data.sort(key=lambda x: float(x[1]), reverse=True)

    print(tabulate.tabulate(summary_data,
                   headers=['Dataset', 'F1-score', 'Std Dev', 'LR', 'Max Depth', 'L2 Reg', 'Max Iter'],
                   tablefmt='grid',
                   floatfmt=('', '.3f', '.3f', '.3f', '', '.2f', ''))) # Formattazione specifica

# Lettura dei file

In [ ]:
# Eseguo il training per tutti i dataset
results_per_dataset = {}

for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Stampo i risultati con tabulate
print_results(results_per_dataset)